# E-Commerce RAG Assistant with Query Routing & OpenTelemetry Tracing

---

An end-to-end **Retrieval-Augmented Generation (RAG)** assistant built for **Fashion Forward Hub**, an e-commerce clothing platform with **44,423 product records** and a store policy **FAQ knowledge base** indexed in **Weaviate**.

### Key Engineering Highlights
1. **Two-Stage LLM Query Router**:
   - **Stage 1 (`check_if_faq_or_product`)**: Routes incoming user queries between the store `FAQ` pipeline and the `Product` catalog pipeline, reducing routing prompt token consumption by **~69%** (~203 tokens $\rightarrow$ ~62 tokens) with 100% test accuracy.
   - **Stage 2 (`decide_task_nature`)**: Classifies product queries as `technical` (exact catalog/price lookups with `temperature=0.3, top_p=0.7`) vs. `creative` (outfit styling recommendations with `temperature=1.0, top_p=0.9`).
2. **Dual-Index Weaviate Retrieval & Progressive Metadata Filtering**:
   - **Semantic FAQ Retrieval (`query_on_faq`)**: Replaces full-FAQ prompt stuffing (`1,116 tokens`) with top-5 semantic vector search (`faq_collection.query.near_text(limit=5)`), reducing FAQ prompt tokens by **~65%** (`386 tokens`).
   - **Product Search & Filter Relaxation (`get_relevant_products_from_query`)**: Supports both structured Weaviate `Filter.by_property(...)` metadata filtering with progressive constraint relaxation and direct zero-LLM-token semantic search (`simplified=True`, saving ~1,465 metadata-generation tokens per query).
3. **LLM Observability & Token Telemetry**:
   - Instrumented with **Arize Phoenix (`phoenix.otel`)** and **OpenTelemetry (`openinference` semantic conventions)** across `tool`, `llm`, `retriever`, and `chain` spans to track prompt/completion token usage, latency bottlenecks, and retrieved document payloads.


# Table of Contents
- [ 1 - Environment Setup & OpenTelemetry Tracing](#1)
  - [ 1.1 Importing Libraries](#1-1)
  - [ 1.2 Connecting to the Weaviate Vector Database](#1-2)
  - [ 1.3 Configuring Arize Phoenix & OpenTelemetry Tracing](#1-3)
- [ 2 - Exploring the Dual Vector Database Schema](#2)
  - [ 2.1 Products Catalog (44,423 Items)](#2-1)
  - [ 2.2 Store FAQ Knowledge Base](#2-2)
- [ 3 - LLM Inference Wrapper & Token Usage Tracking](#3)
  - [ 3.1 Parameter Configuration Helper](#3-1)
- [ 4 - Query Routing & Semantic FAQ Retrieval](#4)
  - [ 4.1 Stage 1 Router: FAQ vs. Product Classification](#4-1)
  - [ 4.2 Formatting FAQ Context](#4-2)
  - [ 4.3 Semantic Search over the Weaviate FAQ Collection](#4-3)
  - [ 4.4 Stage 2 Router: Creative vs. Technical Product Queries](#4-4)
  - [ 4.5 Dynamic LLM Sampling Parameters by Task Nature](#4-5)
- [ 5 - Product Catalog Retrieval & Metadata Filtering](#5)
  - [ 5.1 LLM-Generated Metadata Filters](#5-1)
  - [ 5.2 Connecting to the Weaviate Products Collection](#5-2)
  - [ 5.3 Metadata Filter Construction](#5-3)
  - [ 5.4 Semantic Retrieval vs. Progressive Filter Relaxation](#5-4)
  - [ 5.5 Formatting Retrieved Product Context](#5-5)
  - [ 5.6 End-to-End Product Query Pipeline](#5-6)
- [ 6 - Unified RAG Orchestrator (`answer_query`)](#6)
- [ 7 - Interactive ChatBot & Telemetry Evaluation](#7)


<a id='1'></a>
## 1 - Environment Setup & OpenTelemetry Tracing
---

As user traffic scales on an e-commerce RAG assistant, two production challenges emerge:
1. **Token Cost Escalation**: Stuffing entire FAQ lists or verbose few-shot examples into every LLM router and generation call drives up token consumption.
2. **Latency Bottlenecks**: Generating complex JSON metadata filters via an LLM call prior to vector search adds ~1,500 tokens and extra round-trip latency on every product query.

In this notebook, we instrument the RAG pipeline with **OpenTelemetry + Arize Phoenix** to measure token usage across every span, and systematically optimize each stage (`simplified=True`) to cut token consumption by **60–69%** while maintaining response quality.

<a id='1-1'></a>
### 1.1 Importing Libraries


In [1]:
from weaviate.classes.query import Filter
import weaviate
import joblib

In [2]:
import json
from utils import (
    ChatWidget, 
    generate_with_single_input,
    process_and_print_query,
    print_properties,
    kill_processes_on_ports,
    make_url
)

# Kill processes on ports before importing flask_app and weaviate_server
# WARNING: Running this cell twice may kill the active kernel
kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])

import flask_app
import weaviate_server

 * Serving Flask app 'flask_app'
 * Debug mode: off


<a id='1-2'></a>
### 1.2 Connecting to the Weaviate Vector Database

We connect to the local Weaviate instance hosting the `products` and `Faq` vector collections.


In [3]:
client = weaviate.connect_to_local(port=8079, grpc_port=50050)

<a id='1-3'></a>
### 1.3 Configuring Arize Phoenix & OpenTelemetry Tracing

We initialize Arize Phoenix (`phoenix.otel`) and register an OpenTelemetry tracer provider to record manual spans (`tool`, `llm`, `retriever`, `chain`) along with OpenInference token usage attributes (`llm.token_count.prompt`, `llm.token_count.completion`, `llm.token_count.total`).


In [4]:
import phoenix as px
from phoenix.otel import register
from opentelemetry.trace import Status, StatusCode

In [5]:
# Launch the lab and the URL
make_url()
session = px.launch_app()

FOLLOW THIS URL TO OPEN THE UI: http://rtzatdtomcnp.labs.coursera.org/
🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [22]:
# Setting up the telemetry

phoenix_project_name = "chatbot"

# With phoenix, we just need to register to get the tracer provider with the appropriate endpoint. 

tracer_provider_phoenix = register(project_name=phoenix_project_name, endpoint="http://127.0.0.1:6006/v1/traces")

# Retrieve a tracer for manual instrumentation
tracer = tracer_provider_phoenix.get_tracer(__name__)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: chatbot
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



<a id='2'></a>
## 2 - A Quick Recap on the Database Structure
---

Let's have a quick recap on both the FAQ and Products databases.

Recall:
- Product database: Contains the products and their information.
- FAQ database: Contains the FAQ data.

<a id='2-1'></a>
### 2.1 Products Database

Let's explore the products database that Fashion Forward Hub has available. To make it easier to understand, let's load it as a list of JSON files first.

In [6]:
# Loading products data
products_data = joblib.load('dataset/clothes_json.joblib')

In [7]:
# Let's get one example
products_data[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67.0,
 'product_id': 15970}

The features each product has are:

- **Gender:** Target audience for the product, such as "Men," "Women," or "Unisex."
- **Master Category:** Broad classification like "Apparel" or "Footwear."
- **Sub Category:** Specific category within a master category, such as "Topwear."
- **Article Type:** Exact type of product, e.g., "Shirts" or "Jackets."
- **Base Colour:** Main color of the product, important for customer choice.
- **Season:** Intended season for the product, e.g., "Summer" or "Winter."
- **Year:** Year of release or collection.
- **Usage:** Intended use or occasion, like "Casual" or "Formal."
- **Product Display Name:** Descriptive name used in marketing.
- **Price:** Cost of the product.
- **Product ID:** Unique identifier for managing and tracking inventory.

<a id='2-2'></a>
### 2.2 FAQ Database

Now, let's load the FAQ database and explore it.

In [8]:
faq = joblib.load("dataset/faq.joblib")

In [9]:
# Get an example
faq[:2]

[{'question': 'What are your store hours?',
  'answer': 'Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday.',
  'type': 'general information'},
 {'question': 'Where is Fashion Forward Hub located?',
  'answer': 'Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State.',
  'type': 'general information'}]

The FAQs are organized in a list, where each entry is a dictionary containing the following keys: `question`, `answer`, and `type`.

<a id='3'></a>
## 3 - LLM Inference Wrapper & Token Usage Tracking
---

The helper `generate_with_single_input` returns the full OpenAI-compatible response payload, including `usage.prompt_tokens`, `usage.completion_tokens`, and `usage.total_tokens` so every LLM call can be audited for cost.


Let's now understand the new behavior:

In [10]:
# The output is a dictionary containing the role and content from the LLM call, as well as the token usage.:
result = generate_with_single_input("What are the primary colors?")
print(json.dumps(result, indent = 2))

{
  "id": "p5KxL5W-fdNmn-a4aeef39aca6720c",
  "object": "chat.completion",
  "created": 1792068474,
  "model": "Qwen/Qwen3.5-9B",
  "metadata": {
    "weight_version": "default"
  },
  "prompt": [],
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "The primary colors depend on the medium you are using, as different systems require different sets of colors to create the broadest range of other colors through mixing. However, the two most common systems are **additive** (light) and **subtractive** (pigment/paint).\n\n### 1. Additive Color Mixing (Light)\nThis is the standard for screens, televisions, and digital displays. In this system, colors are created by adding light together.\n*   **Red**\n*   **Green**\n*   **Blue**\n\nOften referred to as **RGB**, mixing these lights in equal amounts creates white light.\n\n### 2. Subtractive Color Mixing (Pigment, Paint, Ink)\nThis is the standard for physical art, traditional printing, an

In [11]:
# To retreive the content, then you should do as follows:
print(result['choices'][0]['message']['content'])

The primary colors depend on the medium you are using, as different systems require different sets of colors to create the broadest range of other colors through mixing. However, the two most common systems are **additive** (light) and **subtractive** (pigment/paint).

### 1. Additive Color Mixing (Light)
This is the standard for screens, televisions, and digital displays. In this system, colors are created by adding light together.
*   **Red**
*   **Green**
*   **Blue**

Often referred to as **RGB**, mixing these lights in equal amounts creates white light.

### 2. Subtractive Color Mixing (Pigment, Paint, Ink)
This is the standard for physical art, traditional printing, and hair dye. Here, colors work by absorbing (subtracting) wavelengths of light.
*   **Red**
*   **Yellow**
*   **Blue**

Often referred to as **RYB**, this is the traditional model taught in elementary art. *Note: In professional color printing (CMYK), the primary colors are actually Cyan, Magenta, and Yellow, with K

In [12]:
# The total tokens count (input + output) for this is:
print(result['usage']['total_tokens'])

404


Note that there is now a new key called `total_tokens`. Usually, LLM costs are measured by token count (both input and output).

You may think this is a complicated way of getting the information, but this is how OpenAI structure works!

<a id='3-1'></a>
### 3.1 Parameter Configuration Helper

`generate_params_dict` packages the prompt, model identifier (`Qwen/Qwen3.5-9B` for fast routing/filtering or `meta-llama/Llama-3.3-70B-Instruct-Turbo` for final generation), and sampling hyperparameters (`temperature`, `top_p`, `max_tokens`).


In [13]:
def generate_params_dict(
    prompt: str,
    temperature: float = 1.0,
    role: str = 'user',
    top_p: float = 1.0,
    max_tokens: int = 500,
    model: str = "Qwen/Qwen3.5-9B"
) -> dict:
    """
    Generates a dictionary of parameters for calling a Language Learning Model (LLM),
    allowing for the customization of several key options that can affect the output from the model. 

    Args:
        prompt (str): The input text that will be provided to the model to guide text generation.
        temperature (float): A value between 0 and 1 that controls the randomness of the model's output; 
            lower values result in more repetitive and deterministic results, while higher values enhance randomness.
        role (str): The role designation to be used in context, typically identifying the initiator of the interaction.
        top_p (float): A value between 0 and 1 that manages diversity through the technique of nucleus sampling; 
            this parameter limits the set of considered words to the smallest possible while maintaining 'top_p' cumulative probability.
        max_tokens (int): The maximum number of tokens that the model is allowed to generate in response, where a token can 
            be as short as one character or as long as one word.
        model (str): The specific model identifier to be utilized for processing the request. This typically specifies both 
            the version and configuration of the LLM to be employed.

    Returns:
        dict: A dictionary containing all specified parameters which can then be used to configure and execute a call to the LLM.
    """
    # Create the dictionary with the necessary parameters
    kwargs = {
        "prompt": prompt,
        "role": role,
        "temperature": temperature,
        "top_p": top_p,
        "max_tokens": max_tokens,
        "model": model
    }
    return kwargs

In [14]:
kwargs = generate_params_dict("Solve 3x^2 + 5 = 0")
print(kwargs)

{'prompt': 'Solve 3x^2 + 5 = 0', 'role': 'user', 'temperature': 1.0, 'top_p': 1.0, 'max_tokens': 500, 'model': 'Qwen/Qwen3.5-9B'}


In [15]:
# Now you can call the LLM 
result = generate_with_single_input(**kwargs)
content = result['choices'][0]['message']['content']
total_tokens = result['usage']['total_tokens']
print(f"Content: {content}\n\nTotal Tokens: {total_tokens}")

Content: To solve the quadratic equation $3x^2 + 5 = 0$, we will isolate the variable $x$ step by step.

### Step-by-Step Deduction

1.  **Start with the given equation:**
    $$3x^2 + 5 = 0$$

2.  **Subtract 5 from both sides** to isolate the term with $x$:
    $$3x^2 = -5$$

3.  **Divide both sides by 3** to get $x^2$ by itself:
    $$x^2 = -\frac{5}{3}$$

4.  **Take the square root of both sides.**
    *   Note: Since the right side is negative ($-\frac{5}{3}$), the solutions will be **imaginary numbers**. We introduce the imaginary unit $i$, where $i = \sqrt{-1}$.
    $$x = \pm\sqrt{-\frac{5}{3}}$$

5.  **Simplify the square root:**
    Separate the negative sign to introduce $i$:
    $$x = \pm i \sqrt{\frac{5}{3}}$$

6.  **Rationalize the denominator:**
    Multiply the numerator and the denominator inside the square root by 3 to remove the fraction from the denominator:
    $$x = \pm i \sqrt{\frac{5 \cdot 3}{3 \cdot 3}}$$
    $$x = \pm i \sqrt{\frac{15}{9}}$$

    Split the squar

<a id='4'></a>
## 4 - Query Routing & Semantic FAQ Retrieval
---

<a id='4-1'></a>
### 4.1 Stage 1 Router: FAQ vs. Product Classification


`check_if_faq_or_product` routes incoming user queries to either the store FAQ workflow or the Product catalog workflow:
- **Baseline (`simplified=False`)**: Uses an 8-example few-shot prompt (~203 tokens per call).
- **Optimized (`simplified=True`)**: Uses a compact zero-shot instruction prompt (~62 tokens per call — a **69% token reduction**) while preserving 100% classification accuracy across the test suite.


In [23]:
def check_if_faq_or_product(query, simplified = False):
    """
    Determines whether a given instruction prompt is related to a frequently asked question (FAQ) or a product inquiry.

    Parameters:
    - query (str): The instruction or query that needs to be labeled as either FAQ or Product related.
    - simplified (bool): If True, uses a simplified prompt.

    Returns:
    - str: The label 'FAQ' if the prompt is deemed a frequently asked question, 'Product' if it is related to product information, or
      None if the label is inconclusive.
    """
 
    # If not simplified, uses a more complex prompt
    if not simplified:
        PROMPT = f"""Label the following instruction as an FAQ related answer or a product related answer for a clothing store.
        Product related answers are answers specific about product information or that needs to use the products to give an answer.
        Examples:
                Is there a refund for incorrectly bought clothes? Label: FAQ
                Where are your stores located?: Label: FAQ
                Tell me about the cheapest T-shirts that you have. Label: Product
                Do you have blue T-shirts under 100 dollars? Label: Product
                What are the available sizes for the t-shirts? Label: FAQ
                How can I contact you via phone? Label: FAQ
                How can I find the promotions? Label: FAQ
                Give me ideas for a sunny look. Label: Product
        Return only one of the two labels: FAQ or Product, nothing more.
        Query to classify: {query}
                 """

    

    # If simlpified, uses a simplified prompt.
    else:
        PROMPT = f"""Classify query as 'FAQ' (policies, sizing, contact, location) or 'Product' (specific items, outfits, prices).
Return exactly one word: FAQ or Product.
Query: {query}"""

        

        
    with tracer.start_as_current_span("routing_faq_or_product", openinference_span_kind = 'tool') as span:
        span.set_input(str({"query":query, "simplified": simplified}))
        
        # Get the kwargs dictinary to call the llm, with PROMPT as prompt, low temperature (0 or near 0) and max_tokens = 10
        kwargs = generate_params_dict(PROMPT, temperature = 0, max_tokens = 10)

        # Call generate_with_single_input with **kwargs
        with tracer.start_as_current_span("router_call", openinference_span_kind = 'llm') as router_span:
            router_span.set_input(kwargs)
            try:
                response = generate_with_single_input(**kwargs) 
            except Exception as error:
                router_span.record_exception(error)
                router_span.set_status(Status(StatusCode.ERROR))
            else:
                # OpenInference Semantic Conventions for computing Costs
                router_span.set_attribute("llm.token_count.prompt", response['usage']['prompt_tokens'])
                router_span.set_attribute("llm.token_count.completion", response['usage']['completion_tokens'])
                router_span.set_attribute("llm.token_count.total", response['usage']['total_tokens'])
                router_span.set_attribute("llm.model_name", response['model'])
                router_span.set_attribute("llm.provider", 'together.ai')
                router_span.set_output(response)
                router_span.set_status(Status(StatusCode.OK))
        
    
        # Get the Label by accessing the content key of the response dictionary
        label = response['choices'][0]['message']['content']
        total_tokens = response['usage']['total_tokens']
        span.set_output(str({"label": label, 'total_tokens':total_tokens}))
        span.set_status(Status(StatusCode.OK))

        # Improvement to prevent cases where LLM outputs more than one word
        if 'faq' in label.lower():
            label = 'FAQ'
        elif 'product' in label.lower():
            label = 'Product'
        else:
            label = 'undefined'
    
        return label, total_tokens

Let's test both versions:

In [25]:
queries = [
    'What is your return policy?', 
    'Give me three examples of blue T-shirts you have available.', 
    'How can I contact the user support?', 
    'Do you have blue Dresses?',
    'Create a look suitable for a wedding party happening during dawn.'
]

labels = ['FAQ', 'Product', 'FAQ', 'Product', 'Product']

for query, correct_label in zip(queries, labels):
    # Call check_if_faq_or_product and store the results
    response_std, tokens_std = check_if_faq_or_product(query, simplified=False)
    response_simp, tokens_simp = check_if_faq_or_product(query, simplified=True)
    
    # Print results
    process_and_print_query(query, correct_label, response_std, tokens_std, response_simp, tokens_simp)

Query: What is your return policy?
  Standard    → Label: FAQ | Tokens: 201
  Simplified  → Label: FAQ | Tokens: 62

Query: Give me three examples of blue T-shirts you have available.
  Standard    → Label: Product | Tokens: 207
  Simplified  → Label: Product | Tokens: 68

Query: How can I contact the user support?
  Standard    → Label: FAQ | Tokens: 203
  Simplified  → Label: FAQ | Tokens: 64

Query: Do you have blue Dresses?
  Standard    → Label: Product | Tokens: 201
  Simplified  → Label: Product | Tokens: 62

Query: Create a look suitable for a wedding party happening during dawn.
  Standard    → Label: Product | Tokens: 207
  Simplified  → Label: Product | Tokens: 68



<a id='4-2'></a>
### 4.2 Formatting FAQ Context

`generate_faq_layout` formats a list of FAQ dictionaries (`question`, `answer`, `type`) into a structured context block for the LLM prompt.


In [26]:
@tracer.tool
def generate_faq_layout(faq_dict):
    """
    Generates a formatted string layout for a list of FAQs.

    This function iterates through a dictionary of frequently asked questions (FAQs) and constructs
    a string where each question is followed by its corresponding answer and type.

    Parameters:
    - faq_dict (list): A list of dictionaries, each containing keys 'question', 'answer', and 'type' 
      representing an FAQ entry.

    Returns:
    - str: A string representing the formatted layout of FAQs, with each entry on a separate line.
    """
    # Initialize an empty string
    t = ""
    
    # Iterate over every FAQ question in the FAQ list
    for f in faq_dict:
        # Append the question with formatted string (remember to use f-string and access the values as f['question'], f['answer'] and so on)
        # Also, do not forget to add a new line character (\n) at the end of each line.
        t += f"Question: {f['question']} Answer: {f['answer']} Type: {f['type']}\n" 

    return t

In [27]:
# You can generate a full faq_layout with the entire FAQ questions
faq_layout = generate_faq_layout(faq)
print(faq_layout[:1000])

Question: What are your store hours? Answer: Our online store is open 24/7. Customer service is available from 9:00 AM to 6:00 PM, Monday through Friday. Type: general information
Question: Where is Fashion Forward Hub located? Answer: Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State. Type: general information
Question: Do you have a physical store location? Answer: At this time, we operate exclusively online. This allows us to offer a broader selection and lower prices directly to you. Type: general information
Question: How can I create an account with Fashion Forward Hub? Answer: Click on 'Sign Up' in the top right corner of our website and follow the instructions to set up your account. Type: general information
Question: How do I subscribe to your newsletter? Answer: To receive the latest updates and promotions, sign up for our newsletter at the bottom of our homepage. Type: general information
Question:

In [28]:
# You can choose some faq questions and generate a layout from them. 
# They just need to be in a list with dictionaries with the necessary keys: 'question', 'answer' and 'type'
print(generate_faq_layout(faq[1:2]))

Question: Where is Fashion Forward Hub located? Answer: Fashion Forward Hub is primarily an online store. Our corporate office is located at 123 Fashion Lane, Trend City, Style State. Type: general information



<a id='4-3'></a>
### 4.3 Semantic Search over the Weaviate FAQ Collection

Rather than injecting the entire FAQ database into every prompt (~1,116 tokens), we batch-index the FAQ entries into a Weaviate `Faq` collection with deterministic UUIDv5 keys and retrieve only the **top-5 semantically relevant FAQs** (`faq_collection.query.near_text(query, limit=5)`).


In [29]:
faq_collection = client.collections.get("Faq")

Let's add the FAQ questions into a collection.

In [30]:
from tqdm import tqdm
from weaviate.util import generate_uuid5
# Set up a batch process with specified fixed size and concurrency
with faq_collection.batch.fixed_size(batch_size=20, concurrent_requests=5) as batch:
    # Iterate over a subset of the dataset
    for document in tqdm(faq):
        # Generate a UUID based on the chunk text for unique identification
        uuid = generate_uuid5(document['question'])

        # Add the chunk object to the batch with properties and UUID
        batch.add_object(
            properties=document,
            uuid=uuid,
        )

100%|██████████| 25/25 [00:00<00:00, 16578.28it/s]


Now you can query them! Let's go over an example.

In [31]:
res = faq_collection.query.near_text("What is the return policy?", limit = 5)

In [32]:
for obj in res.objects:
    print_properties(obj)

{
  "answer": "We accept returns within 30 days of delivery. Conditions apply for specific categories like accessories.",
  "question": "What is your return policy timeframe?",
  "type": "returns and exchanges"
}
{
  "answer": "Sale items are final sale and cannot be returned or exchanged, unless stated otherwise.",
  "question": "Can I return a sale item?",
  "type": "returns and exchanges"
}
{
  "answer": "Return processing typically takes 5-7 business days from when the item is received at our warehouse.",
  "question": "How long does it take to process a return?",
  "type": "returns and exchanges"
}
{
  "answer": "We provide a prepaid return label for domestic returns. For international returns, shipping is at the customer's cost.",
  "question": "Are return shipping costs covered?",
  "type": "returns and exchanges"
}
{
  "answer": "Initiate an exchange through our Returns Center, selecting the item you wish to exchange and the desired replacement.",
  "question": "How do I exchan

#### Comparing Full-FAQ Prompting vs. Top-5 Weaviate Semantic Retrieval

In `query_on_faq`:
- When `simplified=False`, the entire FAQ list is formatted into the prompt (~1,116 tokens).
- When `simplified=True`, Weaviate retrieves the top 5 matching FAQs, logs the retrieved documents to the OpenTelemetry `retriever` span, and reverses their order so the highest-relevance FAQ sits closest to the user query at the bottom of the prompt (~386 tokens — a **65% reduction**).


In [33]:

def query_on_faq(query, simplified = False, **kwargs):
    """
    Constructs a prompt to query an FAQ system and generates a response.

    This function integrates an FAQ layout into the prompt to help generate a suitable answer to the given query
    using a language model. It supports additional keyword arguments to customize the prompt generation process.

    Parameters:
    - query (str): The query about which the function seeks to provide an answer from the FAQ.
    - simplified (bool): If True, uses semantic search to extract a relevant subset of FAQ questions
    - **kwargs: Optional keyword arguments for extra configuration of prompt parameters.

    Returns:
    - str: The response generated from the language model based on the input query and FAQ layout.

    """

    
    # If not simplified, generate the faq layout with the entire FAQ questions
    if not simplified:
        # Set the tracer as a chain type, since in non-simplified version, the full FAQ is used
        with tracer.start_as_current_span("query_on_faq", openinference_span_kind="tool") as span:
            
            span.set_input({"query": query, "simplified": simplified})
            faq_layout = generate_faq_layout(faq)
            
            # Generate the prompt
            PROMPT = f"""You will be provided with an FAQ for a clothing store. 
        Answer the instruction based on it. You might use more than one question and answer to make your answer. Only answer the question and do not mention that you have access to a FAQ. 
        <FAQ_ITEMS>
        PROVIDED FAQ: {faq_layout}
        </FAQ_ITEMS>
        Question: {query}
            """ 
            span.set_attribute("prompt", PROMPT)

            # Generate the parameters dict with PROMPT and **kwargs 
            kwargs = generate_params_dict(PROMPT, **kwargs) 

            span.set_attribute("output", str(kwargs))
            span.set_status(Status(StatusCode.OK))
    
            return kwargs
        
   
    
    else:
        with tracer.start_as_current_span("query_on_faq", openinference_span_kind="tool") as span:
            span.set_input({"query": query, "simplified": simplified})
            with tracer.start_as_current_span("retrieve_faq_questions", openinference_span_kind="retriever") as retrieve_span:
                
                
                
                # Get the 5 most relevant FAQ objects, in this case limit = None
                results = faq_collection.query.near_text(query, limit=5)


                
                # Set the retrieved documents as attributes on the span
                for i, document in enumerate(results.objects): 
                    retrieve_span.set_attribute(f"retrieval.documents.{i}.document.id", str(document.uuid)) 
                    retrieve_span.set_attribute(f"retrieval.documents.{i}.document.metadata", str(document.metadata)) 
                    retrieve_span.set_attribute( 
                        f"retrieval.documents.{i}.document.content", str(document.properties) 
                    )  
            # Transform the results in a list of dictionary
                results = [x.properties for x in results.objects] 
                # Reverse the order to add the most relevant objects in the bottom, so it gets closer to the end of the input
                results.reverse() 
                # Generate the faq layout with the new list of FAQ questions `results`
                faq_layout = generate_faq_layout(results) 

            # Different prompt to deal with this new scenario. 
            PROMPT = (f"You will be provided with a query for a clothing store regarding FAQ. It will be provided relevant FAQ from the clothing store." 
        f"Answer the query based on the relevant FAQ provided. They are ordered in decreasing relevance, so the first is the most relevant FAQ and the last is the least relevant."  
        f"Answer the instruction based on them. You might use more than one question and answer to make your answer. Only answer the question and do not mention that you have access to a FAQ.\n"  
        f"<FAQ>\n"  
        f"RELEVANT FAQ ITEMS:\n{faq_layout}\n"  
        f"</FAQ>\n" 
        f"Query: {query}")

    
        
            span.set_attribute("prompt", PROMPT)
        
            # Generate the parameters dict with PROMPT and **kwargs 
            kwargs = generate_params_dict(PROMPT, **kwargs) 
        
            span.set_attribute("output", str(kwargs))
            span.set_status(Status(StatusCode.OK))
    
            return kwargs

In [35]:
# Get the dictionary of arguments
kwargs = query_on_faq("I received the dress I ordered but I don't like it. How can I return it?")

In [36]:
# The number of split tokens in this prompt is:
print(len(kwargs['prompt'].split()))

788


Note: The number mentioned above doesn’t match the exact token count the model will use. Tokenization is more complex than just splitting by words, so the actual count might be higher. Still, this gives you a rough idea of the text size and is useful for comparison.

In [37]:
# Run the inference
result = generate_with_single_input(**kwargs)

Let's check the content without the simplified version:

In [38]:
print(result['choices'][0]['message']['content'])

To return the dress, you need to initiate an exchange or return through the Returns Center on the website, selecting the item and the desired resolution. Returns are accepted within 30 days of delivery. Domestically, the store provides a prepaid return label, though if the item is a sale item, it is considered final sale and cannot be returned or exchanged unless stated otherwise.


In [39]:
# Get the total tokens
print(result['usage']['total_tokens'])

1116


Note that for one query, the total tokens is around 1220.

Now let's check the simplified version.

In [40]:
# Get the dictionary of arguments
kwargs = query_on_faq("I received the dress I ordered but I don't like it. How can I return it?", simplified = True)

In [41]:
# The number of split tokens in this prompt is:
print(len(kwargs['prompt'].split()))

250


In [42]:
# Run the inference
result = generate_with_single_input(**kwargs)

In [43]:
print(result['choices'][0]['message']['content'])

Since the dress was just delivered, you are within the 30-day return window. To initiate the return, you need to use a prepaid return label provided for domestic returns, and allow 5–7 business days for the processing once your package is received at the warehouse.


In [44]:
# Get the total tokens
print(result['usage']['total_tokens'])

386


Note that the answer is still correct and the final token count is way smaller!

<a id='4-4'></a>
### 4.4 Stage 2 Router: Creative vs. Technical Product Queries

Once a query is routed to the `Product` pipeline, `decide_task_nature` classifies whether the user needs:
- **`technical`**: Factual catalog lookups, price checks, or specific item availability.
- **`creative`**: Outfit styling recommendations or multi-item look composition.

With `simplified=True`, the router prompt is compressed from ~185 tokens down to **~50–71 tokens** (`max_tokens=1`, `temperature=0`).


In [45]:
def decide_task_nature(query, simplified = True):
    """
    Determines the nature of a query, labeling it as either creative or technical.

    This function constructs a prompt for a language model to decide if a given query requires a creative response,
    such as making suggestions or composing ideas, or a technical response, like providing product details or prices.

    Parameters:
    - query (str): The query to be evaluated for its nature.
    - simplified (bool): If True, uses a simplified prompt.

    Returns:
    - str: The label 'creative' if the query requires creative input, or 'technical' if it requires technical information.
    """


    
    if not simplified:
        PROMPT = f"""Decide if the following query is a query that requires creativity (creating, composing, making new things) or technical (information about products, prices etc.). Label it as creative or technical.
          Examples:
          Give me suggestions on a nice look for a nightclub. Label: creative
          What are the blue dresses you have available? Label: technical
          Give me three Tshirts for summer. Label: technical
          Give me a look for attending a wedding party. Label: creative
          Give me suggestions on clothes that match a green Tshirt. Label: creative
          I would like a suggestion on which products match a green Tshirt I already have. Label: creative

          Query to be analyzed: {query}. Only output one token with the label
          """

    # If simplified, uses a simplified query



    else:
        PROMPT = f"""Classify query as 'creative' (making outfits/suggestions) or 'technical' (prices/availability).
Return only one word: creative or technical.
Query: {query}"""


    
    with tracer.start_as_current_span("decide_task_nature", openinference_span_kind="tool") as span:
    # Generate the kwards dictionary by passing the PROMPT, low temperature and max_tokens = 1
        span.set_input({"query":query, "simplified": simplified})
        kwargs = generate_params_dict(PROMPT, temperature = 0, max_tokens = 1)

        with tracer.start_as_current_span("router_call", openinference_span_kind = 'llm') as router_span:
            router_span.set_input(kwargs)
            try:
                response = generate_with_single_input(**kwargs) 
            except Exception as error:
                router_span.record_exception(error)
                router_span.set_status(Status(StatusCode.ERROR))
            else:
                # OpenInference Semantic Conventions for computing Costs
                router_span.set_attribute("llm.token_count.prompt", response['usage']['prompt_tokens'])
                router_span.set_attribute("llm.token_count.completion", response['usage']['completion_tokens'])
                router_span.set_attribute("llm.token_count.total", response['usage']['total_tokens'])
                router_span.set_attribute("llm.model_name", response['model'])
                router_span.set_attribute("llm.provider", 'together.ai')
                router_span.set_output(response)
                router_span.set_status(Status(StatusCode.OK))

        # Get the Label by accessing the content key of the response dictionary
        label = response['choices'][0]['message']['content']
        total_tokens = response['usage']['total_tokens']
        span.set_output(str({"label": label, 'total_tokens':total_tokens}))
        span.set_status(Status(StatusCode.OK))    
    
        return label, total_tokens

In [47]:
queries = ["Give me two sneakers with vibrant colors.",
           "What are the most expensive clothes you have in your catalogue?",
           "I have a green Dress and I like a suggestion on an accessory to match with it.",
           "Give me three trousers with vibrant colors you have in your catalogue.",
           "Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather."
           ]

labels = ['technical', 'technical', 'creative', 'technical', 'creative']

In [48]:
for query, correct_label in zip(queries, labels):
    response, total_tokens = decide_task_nature(query, simplified = True)
    label = response
    if label == correct_label:
        label = "\033[32m" + label + "\033[0m" 
    else:
        label = "\033[31m" + label + "\033[0m"
    if total_tokens > 150:
        total_tokens = "\033[31m"  + str(total_tokens) + "\033[0m"
    else:
        total_tokens = "\033[32m"  + str(total_tokens) + "\033[0m"
    print(f"Query: {query} Label Predicted: {label}. Correct Label: {correct_label} Total Tokens: {total_tokens}")

Query: Give me two sneakers with vibrant colors. Label Predicted: creative. Correct Label: technical Total Tokens: 56
Query: What are the most expensive clothes you have in your catalogue? Label Predicted: technical. Correct Label: technical Total Tokens: 60
Query: I have a green Dress and I like a suggestion on an accessory to match with it. Label Predicted: creative. Correct Label: creative Total Tokens: 66
Query: Give me three trousers with vibrant colors you have in your catalogue. Label Predicted: creative. Correct Label: technical Total Tokens: 61
Query: Create a look for a woman walking in a park on a sunny day. It must be fresh due to hot weather. Label Predicted: creative. Correct Label: creative Total Tokens: 72


<a id='4-5'></a>
### 4.5 Dynamic LLM Sampling Parameters by Task Nature

`get_params_for_task` maps the router's classification to tailored decoding parameters:
- **`technical`**: `temperature=0.3, top_p=0.7` (focused, deterministic catalog answers).
- **`creative`**: `temperature=1.0, top_p=0.9` (diverse styling suggestions).


In [49]:
@tracer.tool
def get_params_for_task(task):
    """
    Retrieves specific language model parameters based on the task nature.

    This function provides parameter sets tailored for creative or technical tasks to optimize
    language model behavior. For creative tasks, higher randomness is encouraged, while technical
    tasks are handled with more focus and precision. A default parameter set is provided for unexpected cases.

    Parameters:
    - task (str): The nature of the task ('creative' or 'technical').

    Returns:
    - dict: A dictionary containing 'top_p' and 'temperature' settings for the specified task.
    """
    # Create the parameters dict for technical and creative tasks
    PARAMETERS_DICT = {"creative": {'top_p': 0.9, 'temperature': 1},
                       "technical": {'top_p': 0.7, 'temperature': 0.3}} 
    
    # If task is technical, return the value for the key technical in PARAMETERS_DICT
    if task == 'technical':
        param_dict = PARAMETERS_DICT['technical'] 

    # If task is creative, return the value for the key creative in PARAMETERS_DICT
    if task == 'creative':
        param_dict = PARAMETERS_DICT['creative'] 

    # If task is a different value, fallback to another set of parameters
    else: # Fallback to a standard value
        param_dict = {'top_p': 0.5, 'temperature': 1} 

    
    return param_dict

<a id='5'></a>
## 5 - Retrieving Items Based on Metadata from a Query

---

In the previous framework, when a query is identified as a product query, you need to find and return relevant products from the vector database. This process works in three main steps:

1. **Generate a metadata JSON** — Use the LLM to guess likely values for some product categories based on the query.
2. **Run a semantic search** — Use those values as filters when querying the database.
3. **Return the results** — Provide the most relevant products found.

The metadata should include values for the following features:

* Gender
* Master Category
* Article Type
* Base Color
* Season
* Usage

These categories offer a good trade-off between being specific enough to improve relevance and general enough to avoid missing results. Using too many or overly detailed filters could lead to no matches, while including too few could make the query too broad and inefficient. This balance helps keep the system fast, accurate, and cost-effective in real-world use.

In [50]:
# Let's remember the data structure of a product
products_data[0]

{'gender': 'Men',
 'masterCategory': 'Apparel',
 'subCategory': 'Topwear',
 'articleType': 'Shirts',
 'baseColour': 'Navy Blue',
 'season': 'Fall',
 'year': 2011,
 'usage': 'Casual',
 'productDisplayName': 'Turtle Check Men Navy Blue Shirt',
 'price': 67.0,
 'product_id': 15970}

This is a dictionary with every possible value for the categories the LLM can pick from to generate a JSON. Note that this dictionary can become huge.

In [51]:
# Run this cell to generate the dictionary with the possible values for each key
values = {}
for d in products_data:
    for key, val in d.items():
        if key in ('product_id', 'price', 'productDisplayName', 'subCategory', 'year'):
            continue
        if key not in values.keys():
            values[key] = set()
        values[key].add(val)

In [52]:
# Example of possible values for the feature 'season'
values['season']

{'All seasons', 'Fall', 'Spring', 'Summer', 'Winter'}

<a id='5-1'></a>
### 5.1 Generate metadata

This function generates a metadata JSON with possible values for each clothing category. The possible values are passed through the dictionary "values". 

Note that the prompt is huge. Let's investigate the total tokens for a query.

In [53]:
def generate_metadata_from_query(query):
    """
    Generates metadata in JSON format based on a given query to filter clothing items.

    This function constructs a prompt for a language model to create a JSON object that will
    guide the filtering of a vector database query for clothing items. It takes possible values from
    a predefined set and ensures only relevant metadata is included in the output JSON.

    Parameters:
    - query (str): The query describing specific clothing-related needs.

    Returns:
    - str: A JSON string representing metadata with keys like gender, masterCategory, articleType,
      baseColour, price, usage, and season. Each value in the JSON is within a list, with prices specified
      as a dict containing "min" and "max" values. Unrestricted keys should use ["Any"] and unspecified
      prices should default to {"min": 0, "max": "inf"}.
    """

    # Set the prompt. Remember to include the query, the desired JSON format, the possible values (passing {values} at some point) 
    # and explain to the LLM what is going on. 
    # Explicitly tell the llm to include gender, masterCategory, ArticleType, baseColour, price, usage and season as keys.
    # Also mention to the llm that price key must be a json with "min" and "max" values (0 if no lower bound and inf if no upper bound)
    # If there is no price set, add min = 0 and max = inf.
    PROMPT = f"""
    One query will be provided. For the given query, there will be a call on vector database to query relevant clothing items. 
    Generate a JSON with useful metadata to filter the products in the query. Possible values for each feature is in the following json: {values}

    Provide a JSON with the features that best fit in the query (can be more than one, write in a list). Also, if present, add a price key, saying if there is a price range (between values, greater than or smaller than some value).
    Only return the JSON, nothing more. price key must be a JSON with "min" and "max" values (0 if no lower bound and inf if no upper bound). 
    Always include gender, masterCategory, articleType, baseColour, price, usage and season as keys. All values must be within lists.
    If there is no price set, add min = 0 and max = inf.
    Only include values that are given in the json above. 
    
    Example of expected JSON:

    {{
    "gender": ["Women"],
    "masterCategory": ["Apparel"],
    "articleType": ["Dresses"],
    "baseColour": ["Blue"],
    "price": {{"min": 0, "max": "inf"}},
    "usage": ["Formal"],
    "season": ["All seasons"]
    }}

    Query: {query}
             """
    with tracer.start_as_current_span("generate_metadata_from_query", openinference_span_kind="tool") as span:
        span.set_input(query)
        with tracer.start_as_current_span("llm_call", openinference_span_kind="llm") as metadata_span:
            # Generate the response with the generate_with_single_input, PROMPT, temperature = 0 (low randomness) and max_tokens = 1500.
            kwargs = {"prompt": PROMPT, 'temperature': 0, "max_tokens": 1500}
            metadata_span.set_input(kwargs)
            try:
                response = generate_with_single_input(**kwargs) 
            except Exception as error:
                metadata_span.record_exception(error)
                metadata_span.set_status(Status(StatusCode.ERROR))
            else:
                # OpenInference Semantic Conventions for computing Costs
                metadata_span.set_attribute("llm.token_count.prompt", response['usage']['prompt_tokens'])
                metadata_span.set_attribute("llm.token_count.completion", response['usage']['completion_tokens'])
                metadata_span.set_attribute("llm.token_count.total", response['usage']['total_tokens'])
                metadata_span.set_attribute("llm.model_name", response['model'])
                metadata_span.set_attribute("llm.provider", 'together.ai')
                metadata_span.set_output(response)
                metadata_span.set_status(Status(StatusCode.OK))

        # Get the Label by accessing the content key of the response dictionary
        content = response['choices'][0]['message']['content']
        total_tokens = response['usage']['total_tokens']
        span.set_output({"content": content, 'total_tokens':total_tokens})
        span.set_status(Status(StatusCode.OK))   

    
    return content, total_tokens

In [54]:
content, total_tokens = generate_metadata_from_query("Create a look for a man that suits a sunny day in the park. I don't want to spend more than 300 dollars on each piece.")

In [55]:
print(content)

{
"gender": ["Men"],
"masterCategory": ["Apparel"],
"articleType": ["Tshirts", "Shorts", "Casual Shoes"],
"baseColour": ["Blue", "Green", "Beige", "White", "Grey", "Black", "Navy Blue"],
"price": {"min": 0, "max": 300},
"usage": ["Casual", "Smart Casual"],
"season": ["Summer"]
}


In [56]:
print(total_tokens)

1465


So far, each product query has involved processing around **1,500 tokens**—mainly because we generate a set of filters across multiple categories before searching.

You will now **simplify** this process.

Instead of creating detailed filters for each category (like gender, color, etc.), the system will just use **semantic search directly on the user query**. This means:

* No more generating metadata.
* Just take the user’s question and run a semantic search on the product collection.

This approach is faster, uses fewer tokens, and is still effective for most queries.

<a id='5-2'></a>
### 5.2 Loading the Weaviate Product Collection

Now it is time to work with the Weaviate collection. It is already given to you and it is the product_data you saw before, but added as a Weaviate collection, so we can query with semantic search and metadata filtering.

In [57]:
products_collection = client.collections.get('products')

In [58]:
len(products_collection)

44423

<a id='5-3'></a>
### 5.3 Metadata Filter Construction

`parse_json_output` and `get_filter_by_metadata` convert the LLM-generated JSON constraints into Weaviate `Filter.by_property(...)` objects (`contains_any` for categorical fields and `greater_than` / `less_than` for `price`).


In [59]:
@tracer.tool
def parse_json_output(llm_output):
    """
    Parses a string output from a language model into a JSON object.

    This function attempts to clean and parse a JSON-formatted string produced by a language model (LLM).
    The input string might contain minor formatting issues, such as unnecessary newlines or single quotes
    instead of double quotes. The function attempts to correct such issues before parsing.

    Parameters:
    - llm_output (str): The string output from the language model that is expected to be in JSON format.

    Returns:
    - dict or None: A dictionary if parsing is successful, or None if the input string cannot be parsed into valid JSON.

    Exception Handling:
    - In case of a JSONDecodeError during parsing, an error message is printed, and the function returns None.
    """
    try:
        # Since the input might be improperly formatted, ensure any single quotes are removed
        llm_output = llm_output.replace("\n", '').replace("'",'').replace("}}", "}").replace("{{", "{")  # Remove any erroneous structures
        
        # Attempt to parse JSON directly provided it is a properly-structured JSON string
        parsed_json = json.loads(llm_output)
        return parsed_json
    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        return None

@tracer.tool
def get_filter_by_metadata(json_output: dict | None = None):
    """
    Generate a list of Weaviate filters based on a provided metadata dictionary.

    Parameters:
    - json_output (dict) or None: Dictionary containing metadata keys and their values.

    Returns:
    - list[Filter] or None: A list of Weaviate filters, or None if input is None.
    """
    # If the input dictionary is None, return None immediately
    if json_output is None:
        return None

    # Define a tuple of valid keys that are allowed for filtering
    valid_keys = (
        'gender',
        'masterCategory',
        'articleType',
        'baseColour',
        'price',
        'usage',
        'season',
    )

    # Initialize an empty list to store the filters
    filters = []

    # Iterate over each key-value pair in the input dictionary
    for key, value in json_output.items():
        # Skip the key if it is not in the list of valid keys
        if key not in valid_keys:
            continue

        # Special handling for the 'price' key
        if key == 'price':
            # Ensure the value associated with 'price' is a dictionary
            if not isinstance(value, dict):
                continue

            # Extract the minimum and maximum prices from the dictionary
            min_price = value.get('min')
            max_price = value.get('max')

            # Skip if either min_price or max_price is not provided
            if min_price is None or max_price is None:
                continue

            # Skip if min_price is non-positive or max_price is infinity
            if min_price <= 0 or max_price == 'inf':
                continue

            # Add filters for price greater than min_price and less than max_price
            filters.append(Filter.by_property(key).greater_than(min_price))
            filters.append(Filter.by_property(key).less_than(max_price))
        else:
            # For other valid keys, add a filter that checks for any of the provided values
            filters.append(Filter.by_property(key).contains_any(value))

    return filters


@tracer.tool
def generate_filters_from_query(query):
    json_string, total_tokens = generate_metadata_from_query(query)
    json_output = parse_json_output(json_string)
    filters = get_filter_by_metadata(json_output)
    return filters, total_tokens

<a id='5-4'></a>
### 5.4 Semantic Retrieval vs. Progressive Filter Relaxation

`get_relevant_products_from_query` supports two retrieval modes over the 44,423-item `products` collection:
1. **Metadata-Filtered Retrieval (`simplified=False`)**: Generates metadata filters via LLM (~1,465 tokens) and runs `products_collection.query.near_text(..., filters=Filter.all_of(filters), limit=20)`. If fewer than 10 products match, it progressively drops lower-priority filters (`importance_order`) inside traced `refilter_i` spans until at least 5 products are found.
2. **Direct Semantic Retrieval (`simplified=True`)**: Bypasses LLM filter generation (`0 LLM tokens`) and directly queries `products_collection.query.near_text(query, limit=20)`, logging all retrieved product UUIDs and properties to OpenTelemetry.


In [60]:

def get_relevant_products_from_query(query, simplified = False):
    """
    Retrieve the most relevant products for a given query by applying semantic search and optional filters.

    This function generates metadata filters from the query and uses them to search for products 
    that best match the intended criteria. If `simplified` is True, it performs only a basic semantic 
    search with no filters. If the filtered search returns too few results, it progressively reduces 
    filtering constraints based on the predefined importance of each filter.

    Parameters:
    query (str): The query string used to search for relevant products.
    simplified (bool): If True, only a simple semantic search is performed without any metadata filters.

    Returns:
    list: A list of product objects that are most relevant to the query.
    total_tokens: The number of tokens used in the LLM call. Returns 0 if simplified search is used.
    """
    
    
    
    # If simplified, just do a semantic search with 20 objects and return it
    if simplified:
        with tracer.start_as_current_span("get_relevant_products_from_query", openinference_span_kind="retriever") as span:  
            span.set_input({'query':query, 'simplified':simplified})
            
            results = products_collection.query.near_text(query, limit=20)

            # Set the retrieved documents as attributes on the span
            for i, document in enumerate(results.objects): 
                span.set_attribute(f"retrieval.documents.{i}.document.id", str(document.uuid)) 
                span.set_attribute(f"retrieval.documents.{i}.document.metadata", str(document.metadata)) 
                span.set_attribute( 
                    f"retrieval.documents.{i}.document.content", str(document.properties)
                )  
    

            
            span.set_output({"results": results.objects, "total_tokens": 0})
            span.set_status(Status(StatusCode.OK))  
    
            return results.objects, 0  # Total tokens in this case is 0 because there was no LLM call!
    # If not simplified, perform the previous workflow by generating the filters and then doing a semantic search with them
    
    with tracer.start_as_current_span("get_relevant_products_from_query", openinference_span_kind="retriever") as span:  
        span.set_input({'query':query, 'simplified':simplified})
        filters, total_tokens = generate_filters_from_query(query)  # Generate filters based on the query

    # Check if there are no applicable filters
        if filters is None or len(filters) == 0:
            span.set_attribute("retrieval.filters", '')
            results = products_collection.query.near_text(query, limit=20) 
            # Set the retrieved documents as attributes on the span
            for i, document in enumerate(results.objects): 
                span.set_attribute(f"retrieval.documents.{i}.document.id", str(document.uuid))
                span.set_attribute(f"retrieval.documents.{i}.document.metadata", str(document.metadata)) 
                span.set_attribute( 
                    f"retrieval.documents.{i}.document.content", str(document.properties) 
                )  
            span.set_output({"results": results.objects, "total_tokens": total_tokens})
            span.set_status(Status(StatusCode.OK))  
            return results.objects, total_tokens
    # Query with filters and limit to the top 20 relevant objects
        span.set_attribute("retrieval.filters", str(filters))
        results = products_collection.query.near_text(query, filters=Filter.all_of(filters), limit=20)
        span.set_attribute("retrieval.len", len(results.objects))
        # Set the retrieved documents as attributes on the span
        for i, document in enumerate(results.objects): 
            span.set_attribute(f"retrieval.documents.{i}.document.id", str(document.uuid))
            span.set_attribute(f"retrieval.documents.{i}.document.metadata", str(document.metadata)) 
            span.set_attribute( 
                f"retrieval.documents.{i}.document.content", str(document.properties) 
            )
    
        # If the result set contains fewer than 10 products, try reducing filters to broaden the search
        importance_order = [ 'baseColour', 'masterCategory', 'usage', 'masterCategory', 'season', 'articleType', 'gender']
        if len(results.objects) < 10:
            # Iterate through the importance order of filters
            for i in range(len(importance_order)):
                with tracer.start_as_current_span(f"refilter_{i}", openinference_span_kind="chain") as refilter_span: 
                    # Create a list of filters that excludes less important ones
                    filtered_filters = [x for x in filters if x.target in importance_order[i+1:]]
                    refilter_span.set_input(str(filtered_filters))
                    
                    results = products_collection.query.near_text(query, filters=Filter.all_of(filtered_filters), limit=20)
                    # Set the retrieved documents as attributes on the span
                    for j, document in enumerate(results.objects): 
                        refilter_span.set_attribute(f"retrieval.documents.{j}.document.id", str(document.uuid))
                        refilter_span.set_attribute(f"retrieval.documents.{j}.document.metadata", str(document.metadata)) 
                        refilter_span.set_attribute( 
                            f"retrieval.documents.{j}.document.content", str(document.properties) 
                        )
                    # If sufficient products have been found, return early
                    if len(results.objects) >= 5:
                        refilter_span.set_output(results.objects)
                        refilter_span.set_status(Status(StatusCode.OK))  
                        span.set_output(results.objects)
                        span.set_status(Status(StatusCode.OK)) 
                        return results.objects, total_tokens
        span.set_output(results.objects)
        span.set_status(Status(StatusCode.OK)) 
        return results.objects, total_tokens  # Return the final set of relevant products

In [61]:
query = "Give me three T-shirts to use in sunny days"

In [ ]:
t, total_tokens = get_relevant_products_from_query(query)

In [ ]:
total_tokens

Around 1500 tokens for this query! Let's try with the simplified version

In [ ]:
t, total_tokens = get_relevant_products_from_query(query, simplified = True)

In [ ]:
total_tokens

Note that this query took 0 tokens, as it didn't use the LLM. It directly used the query to retrieve the objects that are in the vector database.

<a id='5-5'></a>
### 5.5 Formatting Retrieved Product Context

`generate_items_context` serializes the retrieved Weaviate product objects into a structured text block containing `Product ID`, `Product name`, `Product Category`, `Product usage`, `Product gender`, `Product Type`, `Product Color`, `Product Season`, and `Product Year`.


In [ ]:
@tracer.tool
def generate_items_context(results):
    """
    Compile detailed product information from a list of result objects into a formatted string.

    Parameters:
    results (list): A list of result objects, each having a `properties` attribute that is a dictionary 
                    containing product attributes such as 'product_id', 'productDisplayName', 
                    'masterCategory', 'usage', 'gender', 'articleType', 'subCategory', 
                    'baseColour', 'season', and 'year'.

    Returns:
    str: A multi-line string where each line contains the formatted details of a single product.
         Each product detail includes the product ID, name, category, usage, gender, type, color, 
         season, and year.
    """
    t = ""  # Initialize an empty string to accumulate product information

    for item in results:  # Iterate through each item in the results list
        item = item.properties  # Access the properties dictionary of the current item

        # Append formatted product details to the output string
        t += (
            f"Product ID: {item['product_id']}. "
            f"Product name: {item['productDisplayName']}. "
            f"Product Category: {item['masterCategory']}. "
            f"Product usage: {item['usage']}. "
            f"Product gender: {item['gender']}. "
            f"Product Type: {item['articleType']}. "
            f"Product Category: {item['subCategory']} "
            f"Product Color: {item['baseColour']}. "
            f"Product Season: {item['season']}. "
            f"Product Year: {item['year']}.\n"
        )

    return t  # Return the complete formatted string with product details

In [ ]:
print(generate_items_context(t)[:1000])

<a id='5-6'></a>
### 5.6 End-to-End Product Query Pipeline

`query_on_products` orchestrates task nature classification, parameter selection, Weaviate product retrieval, and final prompt construction, returning both the `kwargs` dictionary and cumulative routing/filtering token count.


In [ ]:
@tracer.tool
def query_on_products(query, simplified = False):
    """
    Execute a product query process to generate a response based on the nature of the query.

    Parameters:
    query (str): The input query string that needs to be analyzed and answered using product data.
    task_nature_prompt_function (func): The prompt function to be used to decide the task nature (if creative of technical)
    simplified (bool): If True, does not use LLM to generate metadata for filtering

    Returns:
    dict: A dictionary of keyword arguments (`kwargs`) containing the prompt and additional settings 
          for creating a response, suitable for input to an LLM or other processing system.
    int: Number of tokens used in the process to create the kwargs dictionary

    Outputs:
    str: The content of the generated response from the LLM based on the provided query and product 
         information.
    """
    total_tokens = 0
    # Determine if the query is technical or creative in nature
    
    query_label, tokens = decide_task_nature(query, simplified = simplified)
    
    # Sum the tokens used to decide the task nature (creative or technical)
    total_tokens += tokens

    # Obtain necessary parameters based on the query type
    parameters_dict = get_params_for_task(query_label)
    
    # Retrieve products that are relevant to the query
    relevant_products, tokens = get_relevant_products_from_query(query, simplified = simplified)
    
    # Sum the tokens used to get relevant products 
    total_tokens += tokens
     
    # Create a context string from the relevant products
    context = generate_items_context(relevant_products)

    # Construct a prompt including product details and the query. Remember to add the context and the query in the prompt, also, ask the LLM to provide the product ID in the answer
    PROMPT = (
        f"Given the available set of clothing products given by: "
        f"CLOTHING PRODUCTS AVAILABLE:\n{context}\n"
        f"Answer the question that follows.\n"
        f"Never use more than 5 clothing products available below to compose your answer.\n"
        f"Provide the item ID in your answers.\n"
        f"The other information might be provided but not necessarily all of them, pick only the relevant ones for the given query.\n"
        
        f"QUERY: {query}"
    )
    
    # Generate kwargs (parameters dict) for parameterized input to the LLM with , Prompt, role = 'assistant' and **parameters_dict
    kwargs = generate_params_dict(PROMPT, role='assistant', **parameters_dict) 

    
    return kwargs, total_tokens

Let's check with both the previous setup and the enhanced setup


#### Previous setup with simplified = False

In [ ]:
kwargs, total_tokens = query_on_products('Make a wonderful look for a man attending a wedding party happening during night.', simplified = False)

In [ ]:
result = generate_with_single_input(**kwargs)
print(result['choices'][0]['message']['content'])

Now let's sum the total tokens to generate the kwargs dictionary and the total tokens used in the final execution.

In [ ]:
print(f"Total tokens used in the query is: {total_tokens + result['usage']['total_tokens']}")

**New setup with <code>simplified = True</code>**

In [ ]:
kwargs, total_tokens = query_on_products('Make a wonderful look for a man attending a wedding party happening during night.', simplified = True)

In [ ]:
total_tokens

In [ ]:
result = generate_with_single_input(**kwargs)
print(result['choices'][0]['message']['content'])

In [ ]:
print(f"Total tokens used in the query is: {total_tokens + result['usage']['total_tokens']}")

And the total tokens used in one query was way lower than before!

<a id='6'></a>
## 6 - The final function! 
---
<a id='6-1'></a>
### 6.1 The function to rule them all

Now let's consolidate the functions

The function will:

1. Check if the query is FAQ or Product
2. If FAQ, runs the FAQ related workflow
3. If Product, runs the Product related workflow
4. Add the information into a dataframe

It returns the kwargs dict with the appropriate arguments and the total tokens used to get to the kwargs dict.

In [ ]:
@tracer.tool
def answer_query(query, model = "meta-llama/Llama-3.3-70B-Instruct-Turbo",simplified=False):
    """
    Processes a user's query to determine its type (FAQ or Product) and executes the appropriate workflow.
    
    Parameters:
    - query (str): The query string provided by the user.
    - model (str): The model that will answer the question. Defaults to meta-llama/Llama-3.3-70B-Instruct-Turbo'
    - simplified (bool): If True, uses a simplified version of the method. Defaults to False.
    
    Returns:
    - dict: A dictionary containing keyword arguments for further processing.
      If the query is neither FAQ nor Product-related, returns a default response dictionary instructing
      the assistant to answer based on existing context.
    """
    # Initialize the total tokens used to zero
    total_tokens = 0
    
    # Determine if the query is FAQ or Product and get the token count for this step
    label, tokens = check_if_faq_or_product(query, simplified=simplified)
    
    # Sum the tokens
    total_tokens += tokens
    
    # If the query is neither FAQ nor Product, return a default response
    if label not in ['FAQ', 'Product']:
        return {
            "role": "assistant",
            "prompt": (f"User provided a question that does not fit FAQ or Product-related categories. "
                       f"Answer it based on the context you already have. Query provided by the user: {query}")
        }
    
    # Process the query based on its label
    if label == 'FAQ':
        # Handle FAQ-related queries
        kwargs = query_on_faq(query, simplified=simplified)
    elif label == 'Product':
        try:
            # Handle Product-related queries, with error handling in place
            kwargs, tokens = query_on_products(query, simplified=simplified)
            # Add the tokens to the total tokens
            total_tokens += tokens
        except Exception:
            # Return an error response if an exception occurs during querying
            return {
                "role": "assistant",
                "prompt": (f"User provided a question that broke the querying system. "
                           f"Instruct them to rephrase it. Answer it based on the context you already have. "
                           f"Query provided by the user: {query}")
            }, total_tokens
    # Set the model to answer the final query - usually a better one         
    kwargs['model'] = model
    # Return the kwargs and total_tokens for further processing
    return kwargs, total_tokens

In [ ]:
kwargs, total_tokens = answer_query("Give me three examples of blue t-shirts available on your catalogue.", simplified = False)

In [ ]:
result = generate_with_single_input(**kwargs)
print(result['choices'][0]['message']['content'])

In [ ]:
# To get the total tokens for the call, we must sum the total_tokens to get the kwargs dictionary + total tokens from the LLM call
total_tokens +  result['usage']['total_tokens']

In [ ]:
kwargs, total_tokens = answer_query("Give me three examples of blue t-shirts available on your catalogue.", simplified = True)

In [ ]:
result = generate_with_single_input(**kwargs)
print(result['choices'][0]['message']['content'])

In [ ]:
total_tokens +  result['usage']['total_tokens']

<a id='7'></a>
## 7 - Interactive ChatBot & Telemetry Evaluation

We launch two instances of the interactive `ChatWidget` — one running the baseline pipeline (`simplified=False`) and one running the cost-optimized pipeline (`simplified=True`) — with full trace logging to Arize Phoenix.


In [ ]:
chat_widget_standard = ChatWidget(generator_function = lambda x: answer_query(x, simplified = False), tracer = tracer)

Now go to the UI to check!


Now test the simplified version!

In [ ]:
chat_widget_simplified = ChatWidget(generator_function = lambda x: answer_query(x, simplified = True), tracer = tracer)

---
### Summary of Optimization Results
| Pipeline Component | Baseline (`simplified=False`) | Optimized (`simplified=True`) | Token Savings |
| :--- | :---: | :---: | :---: |
| **Stage 1 Router (`check_if_faq_or_product`)** | ~203 tokens | ~62–68 tokens | **~69% reduction** |
| **FAQ Retrieval (`query_on_faq`)** | 1,116 tokens (Full FAQ) | 386 tokens (Top-5 Weaviate `near_text`) | **~65% reduction** |
| **Stage 2 Router (`decide_task_nature`)** | ~185 tokens | ~50–71 tokens | **~65% reduction** |
| **Product Retrieval (`get_relevant_products_from_query`)** | ~1,465 LLM tokens (Metadata JSON) | 0 LLM tokens (Direct Weaviate `near_text`) | **100% filter-token savings** |
| **End-to-End Product Query (`answer_query`)** | ~2,929 total tokens | ~1,303 total tokens | **~55% overall reduction** |
